# IFEMooring postprocessing

This notebook loads a saved Muscade run, rebuilds just enough context to reuse the helper plotting functions, and then presents the main diagnostics in a compact order:

1. load and inspect the serialized output
2. rebuild the run context used by the postprocessing helpers
3. summarize the inverse solution
4. compare Muscade against the SIMA inputs and inspect the boundary dofs

In [157]:
using Serialization
using Muscade, StaticArrays, GLMakie, Interpolations, LinearAlgebra, CSV, DataFrames, Statistics

include("BiasedStrainGauge.jl")
include("MeshLineGauge.jl")
include("ChainLink.jl")
include("PostProcessHelpers.jl")

root = raw"c:\Users\antoingr\OneDrive - NTNU\Documents\Code\IFEMooring"
cd(root)

figdir = joinpath(root, "figs")
state_path = joinpath(root, "saveddata", "nosoil_180_100perc.bin")

"c:\\Users\\antoingr\\OneDrive - NTNU\\Documents\\Code\\IFEMooring\\saveddata\\nosoil_180_100perc.bin"

## Analysis window basis

Set any of these values to override the default plot window and axis padding. Leave them as `nothing` to keep the full inverse-load range and the standard y-padding.

In [138]:
analysis_window_basis = (
    tmin = 120,
    tmax = 180,
    ylim_padding_ratio = 0.08,
)

(tmin = 120, tmax = 180, ylim_padding_ratio = 0.08)

## Load the saved trajectory

The saved artifact may be a single `State`, a state trajectory, or the full continuation container. The next cell normalizes that shape and prints a short inspection summary so the rest of the notebook can stay simple.

In [158]:
function unwrap_states(obj)
    if obj isa Muscade.State
        return [obj]
    elseif obj isa AbstractVector
        nonmissing = [item for item in obj if item !== nothing]
        isempty(nonmissing) && error("The saved object is an empty vector")
        firstitem = nonmissing[1]
        if firstitem isa Muscade.State
            return nonmissing
        end
        lastitem = nonmissing[end]
        if lastitem isa AbstractVector && !isempty(lastitem)
            return lastitem[1] isa Muscade.State ? lastitem : [lastitem]
        elseif lastitem isa Tuple && !isempty(lastitem)
            candidate = lastitem[1]
            return candidate isa Muscade.State ? [candidate] : [lastitem]
        else
            error("Unsupported saved object shape: $(typeof(lastitem))")
        end
    else
        error("Unsupported saved object type: $(typeof(obj))")
    end
end

loaded = deserialize(state_path)
states = unwrap_states(loaded)
final_state = states[end]

println("Loaded object type: ", typeof(loaded))
println("Normalized trajectory length: ", length(states))
println("Final state type: ", typeof(final_state))

Loaded object type: Vector{Muscade.State{1, 3, 1, @NamedTuple{γ::Float64, iter::Int64}}}
Normalized trajectory length: 181
Final state type: Muscade.State{1, 3, 1, @NamedTuple{γ::Float64, iter::Int64}}


## Rebuild the run context

This cell mirrors only the data needed by the postprocessing helpers: the SIMA CSV inputs, the prescribed motion interpolants, and the line/node topology used when the states were generated.

In [159]:
g = 9.81
ρ = 1025.0
static_bias = 0.0015
state_times = getfield.(states, :time)
Δtᵢₙᵥ = length(state_times) > 1 ? median(diff(state_times)) : 1.0
nsteps = max(length(state_times) - 1, 0)
staticLoadSteps = -10:0.1:0
inverseLoadSteps = state_times
tmin = something(analysis_window_basis.tmin, first(inverseLoadSteps))
tmax = something(analysis_window_basis.tmax, last(inverseLoadSteps))

function time_window_indices(times, tmin = first(times), tmax = last(times))
    return findall((times .>= tmin) .& (times .<= tmax))
end

function padded_limits(series...; padding_ratio = analysis_window_basis.ylim_padding_ratio)
    values = Float64[]
    for series_item in series
        append!(values, collect(skipmissing(vec(series_item))))
    end
    finite_values = values[isfinite.(values)]
    isempty(finite_values) && return (-1.0, 1.0)
    lo = minimum(finite_values)
    hi = maximum(finite_values)
    span = hi - lo
    pad = span > 0 ? max(span * padding_ratio, eps(Float64)) : 1.0
    return (lo - pad, hi + pad)
end

nel = [5, 23, 12, 7]
chain_elements_per_line = [nel[1] - 1, nel[2], nel[3], nel[4]]
chain_elements_total = sum(chain_elements_per_line)

println("Inferred inverse steps: ", length(inverseLoadSteps), " (nsteps = ", nsteps, ", Δtᵢₙᵥ = ", Δtᵢₙᵥ, ")")
println("Time window: [", tmin, ", ", tmax, "]")

file_path = joinpath(root, "input", "test_wo.csv")
file_path_w = joinpath(root, "input", "test_w.csv")
df = CSV.read(file_path, DataFrame; delim=',')
df_w = CSV.read(file_path_w, DataFrame; delim=',')

statX1 = df[1, "Xfairlead1 [m]"]
statY1 = df[1, "Yfairlead1 [m]"]
statZ1 = df[1, "Zfairlead1 [m]"]
statX2 = df[1, "Xfairlead2 [m]"]
statY2 = df[1, "Yfairlead2 [m]"]
statZ2 = df[1, "Zfairlead2 [m]"]
statX3 = df[1, "Xfairlead3 [m]"]
statY3 = df[1, "Yfairlead3 [m]"]
statZ3 = df[1, "Zfairlead3 [m]"]

nSamples = nrow(df)
ratioSampleTaper = 0.02
taper = floor(Int, ratioSampleTaper * nSamples)
ramp = vcat(LinRange(0.0, 1.0, taper), ones(nSamples - 2taper), LinRange(1.0, 0.0, taper))

prescribed_disp_interp = [
    [
        linear_interpolation(vcat(df[1, "time"] - 10.0, df[:, "time"], df[end, "time"] + 10.0), vcat(0.0, (df[:, "Xfairlead1 [m]"].- statX1) .* ramp, 0.0)),
        linear_interpolation(vcat(df[1, "time"] - 10.0, df[:, "time"], df[end, "time"] + 10.0), vcat(0.0, (df[:, "Yfairlead1 [m]"].- statY1) .* ramp, 0.0)),
        linear_interpolation(vcat(df[1, "time"] - 10.0, df[:, "time"], df[end, "time"] + 10.0), vcat(0.0, (df[:, "Zfairlead1 [m]"].- statZ1) .* ramp, 0.0)),
    ],
    [
        linear_interpolation(vcat(df[1, "time"] - 10.0, df[:, "time"], df[end, "time"] + 10.0), vcat(0.0, (df[:, "Xfairlead2 [m]"].- statX2) .* ramp, 0.0)),
        linear_interpolation(vcat(df[1, "time"] - 10.0, df[:, "time"], df[end, "time"] + 10.0), vcat(0.0, (df[:, "Yfairlead2 [m]"].- statY2) .* ramp, 0.0)),
        linear_interpolation(vcat(df[1, "time"] - 10.0, df[:, "time"], df[end, "time"] + 10.0), vcat(0.0, (df[:, "Zfairlead2 [m]"].- statZ2) .* ramp, 0.0)),
    ],
    [
        linear_interpolation(vcat(df[1, "time"] - 10.0, df[:, "time"], df[end, "time"] + 10.0), vcat(0.0, (df[:, "Xfairlead3 [m]"].- statX3) .* ramp, 0.0)),
        linear_interpolation(vcat(df[1, "time"] - 10.0, df[:, "time"], df[end, "time"] + 10.0), vcat(0.0, (df[:, "Yfairlead3 [m]"].- statY3) .* ramp, 0.0)),
        linear_interpolation(vcat(df[1, "time"] - 10.0, df[:, "time"], df[end, "time"] + 10.0), vcat(0.0, (df[:, "Zfairlead3 [m]"].- statZ3) .* ramp, 0.0)),
    ],
]

function reconstruct_topology_from_state(state)
    model = state.model
    wrapperBucket = model.ele[1]
    chainBucket = model.ele[2]
    length(chainBucket) % chain_elements_total == 0 || error("Unexpected chain bucket length: $(length(chainBucket))")
    nlines = min(length(wrapperBucket), length(chainBucket) ÷ chain_elements_total)

    topNodes = Vector{Muscade.NodID}(undef, nlines)
    anodeLists = Vector{Muscade.NodID}(undef, nlines)
    nodeLists = Vector{Vector{Vector{Muscade.NodID}}}(undef, nlines)
    elementLists = Vector{Vector{Muscade.EleID}}(undef, nlines)

    segment_offsets = cumsum(vcat(0, chain_elements_per_line))

    for iline in 1:nlines
        line_start = (iline - 1) * chain_elements_total + 1
        line_chain = chainBucket[line_start:(line_start + chain_elements_total - 1)]
        gauge = wrapperBucket[iline]

        segment1_chain = line_chain[(segment_offsets[1] + 1):segment_offsets[2]]
        segment2_chain = line_chain[(segment_offsets[2] + 1):segment_offsets[3]]
        segment3_chain = line_chain[(segment_offsets[3] + 1):segment_offsets[4]]
        segment4_chain = line_chain[(segment_offsets[4] + 1):segment_offsets[5]]

        topNodes[iline] = gauge.nodID[1]
        anodeLists[iline] = gauge.nodID[end]
        nodeLists[iline] = [
            vcat(gauge.nodID[1], segment1_chain[1].nodID[1], [element.nodID[2] for element in segment1_chain]...),
            [element.nodID[2] for element in segment2_chain],
            [element.nodID[2] for element in segment3_chain],
            [element.nodID[2] for element in segment4_chain],
        ]
        elementLists[iline] = [gauge.ID, [element.ID for element in line_chain]...]
    end

    return model, topNodes, nodeLists, elementLists, anodeLists
end

loaded = deserialize(state_path)
states = unwrap_states(loaded)
final_state = states[end]
mesh_model, topNodes, nodeLists, elementLists, anodeLists = reconstruct_topology_from_state(final_state)

println("Loaded object type: ", typeof(loaded))
println("Normalized trajectory length: ", length(states))
println("Final state type: ", typeof(final_state))
println("Recovered model type: ", typeof(mesh_model))
println("Recovered lines: ", length(topNodes))
println("Nodes per line (segment counts, first line): ", length.(nodeLists[1]))
println("Mesh nodes per line (first line): ", sum(length.(nodeLists[1])))
println("Elements per line (first line): ", length(elementLists[1]))

Inferred inverse steps: 181 (nsteps = 180, Δtᵢₙᵥ = 1.0)
Time window: [120, 180]
Loaded object type: Vector{Muscade.State{1, 3, 1, @NamedTuple{γ::Float64, iter::Int64}}}
Normalized trajectory length: 181
Final state type: Muscade.State{1, 3, 1, @NamedTuple{γ::Float64, iter::Int64}}
Recovered model type: Model
Recovered lines: 3
Nodes per line (segment counts, first line): [6, 23, 12, 7]
Mesh nodes per line (first line): 48
Elements per line (first line): 47


## Reconstructed geometry

A quick topology sanity check, followed by a final-state snapshot of the reconstructed model.

In [141]:
nlines = length(topNodes)
line_colors = [:red, :blue, :green, :orange]
waterDepth = 200.0

geometry_summary = DataFrame(
    line = collect(1:nlines),
    top_node = topNodes,
    anode = anodeLists,
    segment_node_counts = [join(string.(length.(nodeLists[iline])), " / ") for iline in 1:nlines],
    mesh_node_count = [sum(length.(nodeLists[iline])) for iline in 1:nlines],
    element_count = [length(elementLists[iline]) for iline in 1:nlines],
)

display(geometry_summary)

function geometry_xy_limits(state; padding_ratio = 0.12)
    coords = [coord([node])[1] for node in state.model.nod]
    xs = [coord_value[1] for coord_value in coords]
    ys = [coord_value[2] for coord_value in coords]
    xmin, xmax = extrema(xs)
    ymin, ymax = extrema(ys)
    xpad = max((xmax - xmin) * padding_ratio, 1.0)
    ypad = max((ymax - ymin) * padding_ratio, 1.0)
    return xmin - xpad, xmax + xpad, ymin - ypad, ymax + ypad
end

geometry_fig = Figure(size = (2200, 1200), fontsize = 30)
ax_geometry = Axis3(geometry_fig[1, 1], title = "Reconstructed final state", titlesize = 34, xlabel = "X [m]", ylabel = "Y [m]", zlabel = "Z [m]", xlabelsize = 28, ylabelsize = 28, zlabelsize = 34, xlabeloffset = 24, ylabeloffset = 24, zlabeloffset = 60, xgridvisible = false, ygridvisible = false, zgridvisible = false, aspect = (1, 1, 0.3), xticklabelsize = 22, yticklabelsize = 22, zticklabelsize = 20, zticklabelpad = 10)
draw!(ax_geometry, final_state)
xlo, xhi, ylo, yhi = geometry_xy_limits(final_state)
xlims!(ax_geometry, xlo, xhi)
ylims!(ax_geometry, ylo, yhi)
zlims!(ax_geometry, -waterDepth - 20, 10)
ax_geometry.azimuth[] = -pi / 2 + 12*pi/180
ax_geometry.elevation[] = 0 + 8*pi/180

save(joinpath(figdir, "postprocess_geometry.png"), geometry_fig)
geometry_fig

Row,line,top_node,anode,segment_node_counts,mesh_node_count,element_count
,Int64,NodID,NodID,String,Int64,Int64
1,1,NodID(1),NodID(2),6 / 23 / 12 / 7,48,47
2,2,NodID(50),NodID(51),6 / 23 / 12 / 7,48,47
3,3,NodID(99),NodID(100),6 / 23 / 12 / 7,48,47


## Muscade vs SIMA comparison

This section reuses the saved-state extraction helpers to compare the fairlead motions and top-chain tension histories against the CSV inputs.

In [142]:
nlines = length(topNodes)
line_colors = [:red, :blue, :green]
loadSteps = inverseLoadSteps
ramp_end_time = df[taper, :time]
plot_idx = time_window_indices(loadSteps, tmin, tmax)
plotSteps = loadSteps[plot_idx]

function collect_node_series(states, node_ids, field)
    hcat([vec(getdof(states; class = :U, field = field, nodID = [node_id])) for node_id in node_ids]...)
end

muscade_displacements = [collect_node_series(states, topNodes, field) for field in (:upt1, :upt2, :upt3)]

x_series = Vector{AbstractVector}()
y_series = Vector{AbstractVector}()
z_series = Vector{AbstractVector}()

legend_handles = Any[]
legend_labels = String[]

comparison_fig = Figure(size = (1500, 1350), fontsize = 24)
Label(comparison_fig[1, 1], "Top displacements from SIMA and Muscade", fontsize = 30, tellwidth = false)
ax_x = Axis(comparison_fig[3, 1], ylabel = "Top x disp. [m]", ylabelsize = 24, xticklabelsize = 18, yticklabelsize = 18)
ax_y = Axis(comparison_fig[4, 1], ylabel = "Top y disp. [m]", ylabelsize = 24, xticklabelsize = 18, yticklabelsize = 18)
ax_z = Axis(comparison_fig[5, 1], ylabel = "Top z disp. [m]", xlabel = "Time [s]", xlabelsize = 24, ylabelsize = 24, xticklabelsize = 18, yticklabelsize = 18)

for iline in 1:nlines
    color = line_colors[iline]
    xMotion, yMotion, zMotion = prescribed_disp_interp[iline]

    x_prescribed = collect(xMotion(plotSteps))
    y_prescribed = collect(yMotion(plotSteps))
    z_prescribed = collect(zMotion(plotSteps))

    x_muscade = muscade_displacements[1][plot_idx, iline]
    y_muscade = muscade_displacements[2][plot_idx, iline]
    z_muscade = muscade_displacements[3][plot_idx, iline]

    push!(x_series, x_prescribed)
    push!(x_series, x_muscade)
    push!(y_series, y_prescribed)
    push!(y_series, y_muscade)
    push!(z_series, z_prescribed)
    push!(z_series, z_muscade)

    sima_x = lines!(ax_x, plotSteps, x_prescribed, color = color, linestyle = :dot, linewidth = 2.5, label = "SIMA $iline")
    scatter!(ax_x, plotSteps, x_prescribed, color = color, markersize = 9)
    muscade_x = lines!(ax_x, plotSteps, x_muscade, color = color, linestyle = :solid, linewidth = 2.5, label = "Muscade $iline")
    scatter!(ax_x, plotSteps, x_muscade, color = color, markersize = 7)

    push!(legend_handles, sima_x)
    push!(legend_labels, "SIMA $iline")
    push!(legend_handles, muscade_x)
    push!(legend_labels, "Muscade $iline")

    lines!(ax_y, plotSteps, y_prescribed, color = color, linestyle = :dot, linewidth = 2.5, label = "SIMA $iline")
    scatter!(ax_y, plotSteps, y_prescribed, color = color, markersize = 9)
    lines!(ax_y, plotSteps, y_muscade, color = color, linestyle = :solid, linewidth = 2.5, label = "Muscade $iline")
    scatter!(ax_y, plotSteps, y_muscade, color = color, markersize = 7)

    lines!(ax_z, plotSteps, z_prescribed, color = color, linestyle = :dot, linewidth = 2.5, label = "SIMA $iline")
    scatter!(ax_z, plotSteps, z_prescribed, color = color, markersize = 9)
    lines!(ax_z, plotSteps, z_muscade, color = color, linestyle = :solid, linewidth = 2.5, label = "Muscade $iline")
    scatter!(ax_z, plotSteps, z_muscade, color = color, markersize = 7)
end

xlo, xhi = tmin, tmax
ylo, yhi = padded_limits(x_series...)
zlo, zhi = padded_limits(z_series...)
xlims!(ax_x, xlo, xhi)
xlims!(ax_y, xlo, xhi)
xlims!(ax_z, xlo, xhi)
ylims!(ax_x, ylo, yhi)
ylims!(ax_y, ylo, yhi)
ylims!(ax_z, zlo, zhi)

ramp_handle = vlines!(ax_x, [ramp_end_time], color = :black, linewidth = 2.5)
vlines!(ax_y, [ramp_end_time], color = :black, linewidth = 2.5)
vlines!(ax_z, [ramp_end_time], color = :black, linewidth = 2.5)
push!(legend_handles, ramp_handle)
push!(legend_labels, "Ramp slope end")

comparison_legend = Legend(comparison_fig[2, 1], legend_handles, legend_labels, orientation = :horizontal)

save(joinpath(figdir, "postprocess_comparison.png"), comparison_fig)
comparison_fig

## Boundary dof diagnostics

The last block splits the top and anchor dofs so you can check how each line is being constrained during the inverse reconstruction.

In [143]:
nlines = length(topNodes)
line_colors = [:red, :blue, :green]
window_idx = time_window_indices(inverseLoadSteps, tmin, tmax)
windowSteps = inverseLoadSteps[window_idx]

function collect_node_series(states, node_ids, field)
    hcat([vec(getdof(states; class = :U, field = field, nodID = [node_id])) for node_id in node_ids]...)
end

function find_udof_nodes(state, field)
    node_ids = Muscade.NodID[]
    for node in state.model.nod
        try
            values = getdof(state; class = :U, field = field, nodID = [node.ID])
            !isempty(values) && push!(node_ids, node.ID)
        catch
        end
    end
    return node_ids
end

top_udof_nodes = find_udof_nodes(final_state, :upt1)
anchor_udof_nodes = find_udof_nodes(final_state, :uat1)

UdofsTop = [collect_node_series(states, top_udof_nodes, field) for field in (:upt1, :upt2, :upt3)]
UdofsAnchor = [collect_node_series(states, anchor_udof_nodes, field) for field in (:uat1, :uat2, :uat3)]
println("Top U-dof nodes: ", top_udof_nodes)
println("Anchor U-dof nodes: ", anchor_udof_nodes)

boundary_fig = Figure(size = (1900, 1550), fontsize = 26)
Label(boundary_fig[1, 1:2], "", fontsize = 32, tellwidth = false)
component_labels = ["t1", "t2", "t3"]
legend_handles = Any[]
legend_labels = String[]

for comp in 1:3
    ax_top = Axis(boundary_fig[comp + 2, 1], ylabel = "Top $(component_labels[comp]) [kN]", ylabelsize = 24, xticklabelsize = 18, yticklabelsize = 18)
    ax_anchor = Axis(boundary_fig[comp + 2, 2], ylabel = "Anchor $(component_labels[comp]) [kN]", ylabelsize = 24, xticklabelsize = 18, yticklabelsize = 18)

    top_series = Vector{AbstractVector}()
    anchor_series = Vector{AbstractVector}()

    for iline in 1:nlines
        color = line_colors[iline]
        top_window = UdofsTop[comp][window_idx, iline] ./ 1e3
        anchor_window = UdofsAnchor[comp][window_idx, iline] ./ 1e3
        top_line = lines!(ax_top, windowSteps, top_window, color = color, linewidth = 2.5, label = "Line $iline")
        lines!(ax_anchor, windowSteps, anchor_window, color = color, linewidth = 2.5, label = "Line $iline")

        push!(top_series, top_window)
        push!(anchor_series, anchor_window)

        if comp == 1
            push!(legend_handles, top_line)
            push!(legend_labels, "Line $iline")
        end
    end

    xlims!(ax_top, tmin, tmax)
    xlims!(ax_anchor, tmin, tmax)
    ylims!(ax_top, padded_limits(top_series...))
    ylims!(ax_anchor, padded_limits(anchor_series...))

    if comp == 3
        ax_top.xlabel = "Time [s]"
        ax_anchor.xlabel = "Time [s]"
        ax_top.xlabelsize = 24
        ax_anchor.xlabelsize = 24
    end
end

Legend(boundary_fig[2, 1:2], legend_handles, legend_labels, orientation = :horizontal)

save(joinpath(figdir, "postprocess_boundary_dofs.png"), boundary_fig)
boundary_fig

Top U-dof nodes: Muscade.NodID[Muscade.NodID(1), Muscade.NodID(50), Muscade.NodID(99)]
Anchor U-dof nodes: Muscade.NodID[Muscade.NodID(49), Muscade.NodID(98), Muscade.NodID(147)]


## Axial tension difference

This section compares the reconstructed top-chain axial tension against the SIMA reference curves. The plotted quantity is Muscade minus SIMA, so zero means perfect agreement.

In [160]:
nlines = length(topNodes)
line_colors = [:red, :blue, :green]
loadSteps = inverseLoadSteps
window_idx = time_window_indices(loadSteps, tmin, tmax)
windowSteps = loadSteps[window_idx]

function padded_limits(series...; padding_ratio = analysis_window_basis.ylim_padding_ratio)
    values = Float64[]
    for series_item in series
        append!(values, collect(skipmissing(vec(series_item))))
    end
    finite_values = values[isfinite.(values)]
    isempty(finite_values) && return (-1.0, 1.0)
    lo = minimum(finite_values)
    hi = maximum(finite_values)
    span = hi - lo
    pad = span > 0 ? max(span * padding_ratio, eps(Float64)) : 1.0
    return (lo - pad, hi + pad)
end

line_elements = [elementLists[iline][1] for iline in 1:nlines]
Fgps = extractAxialForceFromStrainGauge(states, line_elements, length(loadSteps))

reference_tensions = [
    linear_interpolation(df[:, "time"], df[:, "TensionTopChainL1 [N]"] ./ 1e3, extrapolation_bc = Flat()),
    linear_interpolation(df[:, "time"], df[:, "TensionTopChainL2 [N]"] ./ 1e3, extrapolation_bc = Flat()),
    linear_interpolation(df[:, "time"], df[:, "TensionTopChainL3 [N]"] ./ 1e3, extrapolation_bc = Flat()),
]

difference_series = [
    vec(Fgps[iline, 1:length(loadSteps)]) ./ 1e3 .- collect(reference_tensions[iline](loadSteps))
    for iline in 1:nlines
]
windowed_difference_series = [difference_series[iline][window_idx] for iline in 1:nlines]

diff_fig = Figure(size = (1500, 1000), fontsize = 24)
Label(diff_fig[1, 1], "Axial tension difference: Muscade - SIMA", fontsize = 30, tellwidth = false)
ax_diff = Axis(diff_fig[3, 1], xlabel = "Time [s]", ylabel = "ΔF [kN]", xlabelsize = 24, ylabelsize = 24, xticklabelsize = 18, yticklabelsize = 18)

handles = Any[]
labels = String[]

for iline in 1:nlines
    handle = lines!(ax_diff, windowSteps, windowed_difference_series[iline], color = line_colors[iline], linewidth = 2.5, label = "Line $iline")
    push!(handles, handle)
    push!(labels, "Line $iline")
end
zero_handle = hlines!(ax_diff, [0.0], color = :black, linestyle = :dash, linewidth = 1.5, label = "Zero difference")
push!(handles, zero_handle)
push!(labels, "Zero difference")

xlims!(ax_diff, tmin, tmax)
ylo, yhi = padded_limits(windowed_difference_series...)
ylims!(ax_diff, ylo, yhi)
Legend(diff_fig[2, 1], handles, labels, orientation = :horizontal)

save(joinpath(figdir, "postprocess_axial_tension_difference.png"), diff_fig)
diff_fig

## Axial tension comparison

This section overlays the reconstructed top-chain tension and the SIMA reference tension for each line. Muscade is drawn as a solid line and SIMA as a dotted line.

In [161]:
nlines = length(topNodes)
line_colors = [:red, :blue, :green]
loadSteps = inverseLoadSteps
window_idx = time_window_indices(loadSteps, tmin, tmax)
windowSteps = loadSteps[window_idx]

function padded_limits(series...; padding_ratio = analysis_window_basis.ylim_padding_ratio)
    values = Float64[]
    for series_item in series
        append!(values, collect(skipmissing(vec(series_item))))
    end
    finite_values = values[isfinite.(values)]
    isempty(finite_values) && return (-1.0, 1.0)
    lo = minimum(finite_values)
    hi = maximum(finite_values)
    span = hi - lo
    pad = span > 0 ? max(span * padding_ratio, eps(Float64)) : 1.0
    return (lo - pad, hi + pad)
end

muscade_tension = [vec(Fgps[iline, 1:length(loadSteps)]) ./ 1e3 for iline in 1:nlines]
reference_tension = [collect(reference_tensions[iline](loadSteps)) for iline in 1:nlines]

compare_fig = Figure(size = (1500, 1250), fontsize = 24)
Label(compare_fig[1, 1], "Axial tension comparison: Muscade vs SIMA", fontsize = 30, tellwidth = false)

function plot_tension_panel!(ax, line_ids)
    panel_handles = Any[]
    panel_labels = String[]

    for iline in line_ids
        color = line_colors[iline]
        muscade_handle = lines!(ax, windowSteps, muscade_tension[iline][window_idx], color = color, linewidth = 2.5, linestyle = :solid, label = "Muscade $iline")
        scatter!(ax, windowSteps, muscade_tension[iline][window_idx], color = color, markersize = 6)
        sima_handle = lines!(ax, windowSteps, reference_tension[iline][window_idx], color = color, linewidth = 2.5, linestyle = :dot, label = "SIMA $iline")
        scatter!(ax, windowSteps, reference_tension[iline][window_idx], color = color, markersize = 6, marker = :circle)

        push!(panel_handles, muscade_handle)
        push!(panel_labels, "Muscade $iline")
        push!(panel_handles, sima_handle)
        push!(panel_labels, "SIMA $iline")
    end

    return panel_handles, panel_labels
end

ax_line1 = Axis(compare_fig[3, 1], title = "Line 1", xlabel = "Time [s]", ylabel = "Axial tension [kN]", xlabelsize = 24, ylabelsize = 24, xticklabelsize = 18, yticklabelsize = 18)
handles1, labels1 = plot_tension_panel!(ax_line1, [1])
xlims!(ax_line1, tmin, tmax)
ylo1, yhi1 = padded_limits(muscade_tension[1][window_idx], reference_tension[1][window_idx])
ylims!(ax_line1, ylo1, yhi1)

ax_line23 = Axis(compare_fig[4, 1], title = "Lines 2 and 3", xlabel = "Time [s]", ylabel = "Axial tension [kN]", xlabelsize = 24, ylabelsize = 24, xticklabelsize = 18, yticklabelsize = 18)
handles23, labels23 = plot_tension_panel!(ax_line23, [2, 3])
xlims!(ax_line23, tmin, tmax)
ylo23, yhi23 = padded_limits(muscade_tension[2][window_idx], muscade_tension[3][window_idx], reference_tension[2][window_idx], reference_tension[3][window_idx])
ylims!(ax_line23, ylo23, yhi23)

legend_handles = vcat(handles1, handles23)
legend_labels = vcat(labels1, labels23)
Legend(compare_fig[2, 1], legend_handles, legend_labels, orientation = :horizontal)

save(joinpath(figdir, "postprocess_axial_tension_comparison.png"), compare_fig)
compare_fig